# MIRAGE++ — Notebook 5: Evaluation and Visualisation

This notebook is a diagnostic toolkit: it shows how to read every signal the model produces during and after training.

## What to Measure and Why

A MIRAGE++ model produces three orthogonal sources of information:

1. **Optimisation diagnostics** — did the solver converge? Was the learning rate right?
   Tracked via `loss_history`, `entropy_history`, `convergence_summary()`.

2. **Weight quality** — are the weights diverse, interpretable, close to ground truth?
   Tracked via $H(\theta)$, HHI, ENB, cosine similarity.

3. **Predictive quality** — MSE, $R^2$, bias–variance under cross-validation.
   Tracked via standard sklearn metrics across folds.

Together these tell a complete story: a model with low MSE but high HHI is over-concentrated and fragile; a model with high ENB but high MSE has sacrificed too much fit for diversity. The $\lambda$ parameter mediates this tradeoff.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from mirror_linear_regression import MirrorLinearRegression
from mirror_linear_regression.convergence import (
    kl_regret_bound, euclidean_regret_bound, minimax_lower_bound,
    optimal_learning_rate, print_convergence_report
)
from mirror_linear_regression.utils_math import (
    entropy, herfindahl_index, effective_number_of_bets
)

rng = np.random.RandomState(42)
m, n = 300, 15
X = rng.randn(m, n)
w_true = rng.dirichlet(np.ones(n))
y = X @ w_true + 0.02 * rng.randn(m)
print(f'Dataset: m={m}, n={n}')
print(f'True weight entropy: {entropy(w_true):.3f} nats  (max = {np.log(n):.3f})')
print(f'True ENB: {effective_number_of_bets(w_true):.2f}  (max = {n})')

## 1. Convergence Diagnostics

The three history arrays track the decomposition of the loss:

$$\mathcal{L}(\theta_t) = \underbrace{\frac{1}{m}\|X\theta_t - y\|^2}_{\text{MSE}_t} - \underbrace{\lambda H(\theta_t)}_{\text{entropy bonus}_t}$$

Watching these separately diagnoses whether the optimiser is improving fit, increasing diversity, or both. Specifically:

- If MSE is decreasing but entropy is also decreasing, the model is fitting   at the cost of concentration — $\lambda$ may be too small.
- If entropy saturates early, the model reached its maximum diversity given the   data — raising $\lambda$ would over-regularise.
- If loss stops improving after very few iterations, the learning rate may be too large.

In [ ]:
# Fit models with tol=0 to see the full trajectory
optimisers = [
    ('Mirror Descent',   'mirror_descent',   0.10),
    ('Natural Gradient', 'natural_gradient',  0.05),
    ('Mirror Prox',      'mirror_prox',       0.10),
    ('AdaMirror',        'ada_mirror',        0.20),
]
models = {}
for label, opt, lr in optimisers:
    model = MirrorLinearRegression(optimizer=opt, lam=0.05, learning_rate=lr, n_iters=600, tol=0)
    model.fit(X, y)
    models[label] = model

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = ['#2166ac', '#1a9850', '#d6604d', '#762a83']
for (label, _, _), color in zip(optimisers, colors):
    mo = models[label]
    axes[0].semilogy(mo.loss_history,    color=color, lw=1.8, label=label)
    axes[1].plot(mo.entropy_history, color=color, lw=1.8)
    axes[2].plot(mo.mse_history,     color=color, lw=1.8)

for ax, title, ylabel in zip(axes,
    ['Loss (log)', 'Entropy H(theta)', 'MSE component'],
    ['Loss', 'H(theta)', 'MSE']):
    ax.set_xlabel('Iteration'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)
axes[1].axhline(np.log(n), color='gray', ls='--', lw=1, label=f'max = ln({n})')
axes[1].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

print('Convergence summaries:')
for label, mo in models.items():
    cs = mo.convergence_summary()
    print(f'  {label:<20s}  iters={cs["n_iters_run"]:3d}  '
          f'converged={cs["converged"]}  final_H={cs["final_entropy"]:.3f}  '
          f'ENB={cs["final_enb"]:.1f}')

## 2. Lambda Sensitivity: MSE–Diversity Tradeoff

The $\lambda$ parameter has a well-defined meaning in the optimisation landscape: it is the Lagrange multiplier on the entropy constraint in the equivalent constrained problem:

$$\min_{\theta \in \Delta_{n-1}} \frac{1}{m}\|X\theta - y\|^2 \quad \text{s.t.} \quad H(\theta) \geq h^*$$

where $h^*$ is implicitly determined by $\lambda$. Larger $\lambda$ requires higher entropy, pulling toward the uniform distribution.

**Practical guidance**:
- $\lambda \approx 0.001$: near-OLS solution, concentration risk remains
- $\lambda \approx 0.05$: good MSE–diversity tradeoff for most problems
- $\lambda \approx 0.3$: near-uniform weights, diversity dominates fit

Use 5-fold cross-validation to select $\lambda$ from a log-spaced grid.

In [ ]:
lambdas = np.logspace(-3, 0, 25)
kf5 = KFold(n_splits=5, shuffle=True, random_state=0)
cv_mse, h_vals, enb_vals, hhi_vals = [], [], [], []

for lam in lambdas:
    fold_mse = []
    for tr, te in kf5.split(X):
        mo = MirrorLinearRegression(lam=lam, n_iters=400)
        mo.fit(X[tr], y[tr])
        fold_mse.append(mean_squared_error(y[te], mo.predict(X[te])))
    mf = MirrorLinearRegression(lam=lam, n_iters=400)
    mf.fit(X, y)
    cv_mse.append(float(np.mean(fold_mse)))
    h_vals.append(float(entropy(mf.weights)))
    enb_vals.append(float(effective_number_of_bets(mf.weights)))
    hhi_vals.append(float(herfindahl_index(mf.weights)))

opt_lam = lambdas[np.argmin(cv_mse)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
data = [('CV MSE', cv_mse, '#2166ac'), ('Entropy H', h_vals, '#d6604d'),
        ('ENB', enb_vals, '#1a9850'), ('HHI', hhi_vals, '#762a83')]
for ax, (title, vals, c) in zip(axes, data):
    ax.semilogx(lambdas, vals, color=c, lw=2)
    ax.axvline(opt_lam, color='gray', ls='--', lw=1, label=f'opt lambda={opt_lam:.4f}')
    ax.set_xlabel(r'$\lambda$'); ax.set_title(title, fontweight='bold')
    ax.legend(frameon=False, fontsize=8)
axes[2].axhline(n, ls='--', color='gray', lw=1, label=f'max = {n}')
axes[2].legend(frameon=False, fontsize=8)
plt.suptitle(r'$\lambda$ Sensitivity: MSE–Diversity Tradeoff', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Optimal lambda by CV MSE: {opt_lam:.4f}')
print(f'At optimal lambda: ENB={enb_vals[np.argmin(cv_mse)]:.2f}  H={h_vals[np.argmin(cv_mse)]:.3f}')

## 3. Theoretical Convergence Report

MIRAGE++ exposes `print_convergence_report()` which compares the empirical training behaviour against three theoretical benchmarks:

- **KL regret bound** $G\sqrt{2T\log n}$: the upper bound achieved by mirror descent
- **Minimax lower bound** $\frac{G}{2}\sqrt{\frac{T\log n}{2}}$: no algorithm can do better
- **Euclidean bound** $G\sqrt{2nT}$: what Ridge/PGD achieves

The ratio KL/Lower should be approximately 4 (constant), confirming minimax optimality in rate. The Euclidean/Lower ratio grows as $\sqrt{n/\log n}$ — for $n = 15$ it is already $\approx 2.0\times$.

In [ ]:
best_model = MirrorLinearRegression(lam=float(opt_lam), learning_rate=0.1, n_iters=500)
best_model.fit(X, y)
print_convergence_report(best_model.loss_history, dim=n, eta=0.1, lam=float(opt_lam))
print(f'\nFinal weights (n={n}):')
print(np.round(best_model.weights, 4))
print(f'Sum: {best_model.weights.sum():.6f}  Min: {best_model.weights.min():.5f}')

## Summary

The evaluation toolkit for a MIRAGE++ model has four components:

1. **Convergence check**: `loss_history`, `entropy_history`, `convergence_summary()` —    verify the solver converged and the entropy reached the expected level.

2. **Lambda tuning**: 5-fold CV across a log-spaced $\lambda$ grid — pick the    value that minimises CV MSE; check ENB at that point for diversity.

3. **Theoretical bounds**: `print_convergence_report()` confirms the empirical    convergence rate is consistent with $O(\sqrt{T\log n})$ theory.

4. **Weight diagnostics**: `herfindahl_index`, `effective_number_of_bets`,    `entropy` — confirm the solution is genuinely diversified, not just formally constrained.

**Next → Notebook 6**: Full head-to-head comparison of MIRAGE++ vs OLS, Ridge, Lasso across all six synthetic scenarios with statistical significance testing.